# Yield window by GPQR

In [ ]:
import sys
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

sys.path.insert(0, os.path.abspath(".."))

warnings.filterwarnings("ignore")

SLURRIES = ["G50", "G45", "G40", "G40+IPA"]

In [ ]:
X = pd.read_csv("../../_temp/v0/X.csv", index_col=[0, 1, 2])
y = pd.read_csv("../../_temp/v0/y.csv", index_col=[0, 1, 2])
Xpred = pd.read_csv("../../_temp/v0/Xpred_2D.csv", index_col=[0, 1, 2])
Delaunay = pd.read_csv("../../_temp/v0/delaunay.Xpred_2D.csv", index_col=[0, 1, 2])

In [ ]:
columns = ["slurry", "cosine_of_contact_angle"]
cos_thetas = X["cosine_of_contact_angle"].reset_index()[columns]
slurry_map = cos_thetas.drop_duplicates().set_index("cosine_of_contact_angle")["slurry"]

slurries = Xpred["cosine_of_contact_angle"].map(slurry_map)
unique_slurries = [s for s in SLURRIES if s in slurries.unique()]

## Plot

In [ ]:
N_COLORS = 8
cmap = mcolors.LinearSegmentedColormap.from_list(
    "gray_blue",
    [mcolors.to_rgba("gray", alpha=0.3), mcolors.to_rgba("tab:blue", alpha=0.8)],
    N=N_COLORS,
)

levels = np.linspace(0, 1, N_COLORS + 1)
norm = mcolors.BoundaryNorm(levels, ncolors=N_COLORS)

## Marginal probability (H)

In [ ]:
marginal = pd.read_csv(
    "../../_temp/v0/H.marginal.Xpred_2D.csv", index_col=["index", "batch", "sample"]
)
marginal = marginal[marginal["target"] == "H"].drop(columns=["target"])
marginal = marginal.groupby(level=["index"]).mean()

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey=True)

for slurry, ax in zip(unique_slurries, axes):
    ok_pred = slurries == slurry

    this_Xpred = Xpred[ok_pred].to_xarray().to_array().values
    this_prob = marginal[ok_pred.values].values

    delaunay = Delaunay[ok_pred.values].values.reshape(this_Xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    ax.contourf(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        this_prob.reshape(this_Xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.contour(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Marginal probability (H)", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
cbar.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))

fig.supxlabel("Rgt")
fig.supylabel("Ca")

fig.show()

## Marginal probability (phi)

In [ ]:
marginal = pd.read_csv(
    "../../_temp/v0/phi.marginal.Xpred_2D.csv", index_col=["index", "batch", "sample"]
)
marginal = marginal[marginal["target"] == "phi"].drop(columns=["target"])
marginal = marginal.groupby(level=["index"]).mean()

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey=True)

for slurry, ax in zip(unique_slurries, axes):
    ok_pred = slurries == slurry

    this_Xpred = Xpred[ok_pred].to_xarray().to_array().values
    this_prob = marginal[ok_pred.values].values

    delaunay = Delaunay[ok_pred.values].values.reshape(this_Xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    ax.contourf(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        this_prob.reshape(this_Xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.contour(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Marginal probability (phi)", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
cbar.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))

fig.supxlabel("Rgt")
fig.supylabel("Ca")

fig.show()

## Joint probability

In [ ]:
joint = pd.read_csv(
    "../../_temp/v0/joint_probability.Xpred_2D.csv",
    index_col=["index", "batch", "sample"],
)
joint = joint.groupby(level=["index"]).mean()

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey=True)

for slurry, ax in zip(unique_slurries, axes):
    ok_pred = slurries == slurry

    this_Xpred = Xpred[ok_pred].to_xarray().to_array().values
    this_prob = joint[ok_pred.values].values

    delaunay = Delaunay[ok_pred.values].values.reshape(this_Xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    ax.contourf(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        this_prob.reshape(this_Xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.contour(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Joint probability", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
cbar.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))

fig.supxlabel("Rgt")
fig.supylabel("Ca")

fig.show()